# Arquitectura Medallion (Bronze, Silver y Gold)

#### 1. Importamos librerías

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession


#### 2. Establecimiento de rutas

In [0]:
CATALOG = "airline_mantenimiento"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

VOLUME_NAME = "landing"

VUELOS_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}/vuelos_diarios.csv"
AERONAVES_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}/aeronaves.csv"
AEROPUERTOS_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}/aeropuertos.csv"
MANTENIMIENTOS_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}/mantenimientos_rds.csv"

spark = SparkSession.builder.getOrCreate()



#### 3. Creación catálogo, esquemas y volumen

In [0]:
#Creamos el catalogo
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

#Creamos los esquemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

#Creamos el volumen
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}")


DataFrame[]

In [0]:
#Leemos vuelos

vuelos_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(VUELOS_PATH)
)

#Leemos aeronaves
aeronaves_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(AERONAVES_PATH)
)

#Leemos aeropuertos
aeropuertos_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(AEROPUERTOS_PATH)
)

#Leemos mantenimientos
mantenimientos_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(MANTENIMIENTOS_PATH)
)

In [0]:
#Validación 
print("Filas vuelos:", vuelos_raw.count())
print("Filas aeronaves:", aeronaves_raw.count())
print("Filas aeropuertos:", aeropuertos_raw.count())
print("Filas mantenimientos:", mantenimientos_raw.count())


Filas vuelos: 100
Filas aeronaves: 10
Filas aeropuertos: 5
Filas mantenimientos: 100


In [0]:
#Normalizar vuelos Bronze

vuelos_bronze = (
    vuelos_raw
    .withColumn("fecha", F.to_date(F.col("fecha")))
    .withColumn("vuelo_id", F.col("vuelo_id").cast("string"))
    .withColumn("origen_id", F.col("origen_id").cast("string"))
    .withColumn("destino_id", F.col("destino_id").cast("string"))
    .withColumn("aeronave_id", F.col("aeronave_id").cast("string"))
    .withColumn("duracion_min", F.col("duracion_min").cast("int"))
    .withColumn("estado", F.col("estado").cast("string"))
    .withColumn("ingestion_time", F.current_timestamp())
)

#Normalizar aeronaves Bronze

aeronaves_bronze = (
    aeronaves_raw
    .withColumn("aeronave_id", F.col("aeronave_id").cast("string"))
    .withColumn("anio_fabricacion", F.col("anio_fabricacion").cast("int"))
    .withColumn("modelo", F.col("modelo").cast("string"))
    .withColumn("fabricante", F.col("fabricante").cast("string"))
    .withColumn("ingestion_time", F.current_timestamp())
)

#Normalizar aeropuertos Bronze
aeropuertos_bronze = (
    aeropuertos_raw
    .withColumn("aeropuerto_id", F.col("aeropuerto_id").cast("string"))
    .withColumn("nombre", F.col("nombre").cast("string"))
    .withColumn("ciudad", F.col("ciudad").cast("string"))
    .withColumn("pais", F.col("pais").cast("string"))
    .withColumn("lat", F.col("lat").cast("double"))
    .withColumn("lon", F.col("lon").cast("double"))
    .withColumn("ingestion_time", F.current_timestamp())
)
#Normalizar mantenimientos Bronze
mantenimientos_bronze = (
    mantenimientos_raw
    .withColumn("mantenimiento_id", F.col("mantenimiento_id").cast("string"))
    .withColumn("aeronave_id", F.col("aeronave_id").cast("string"))
    .withColumn("fecha", F.to_date(F.col("fecha")))
    .withColumn("tipo", F.col("tipo").cast("string"))
    .withColumn("costo_usd", F.col("costo_usd").cast("double"))
    .withColumn("duracion_hr", F.col("duracion_hr").cast("double"))
    .withColumn("ingestion_time", F.current_timestamp())
)




In [0]:
# Guardar tablas bronze

vuelos_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.vuelos_bronze") 

aeronaves_bronze.write.format("delta") \
.mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.aeronaves_bronze")

aeropuertos_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.aeropuertos_bronze")

mantenimientos_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.mantenimientos_bronze")
aeronaves_bronze.write.mode("overwrite")

#### 4. Capa Silver

In [0]:
#Leemos las tablas bronze
vuelos_bz = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.vuelos_bronze")
aeronaves_bz = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.aeronaves_bronze")
aeropuertos_bz = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.aeropuertos_bronze")
mantenimientos_bz = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.mantenimientos_bronze")


In [0]:
#Transformaciones Silver vuelos
vuelos_slv=(
    vuelos_bz
        .select("vuelo_id", "fecha", "origen_id", "destino_id", "aeronave_id", "estado", "duracion_min")
        .withColumn("fecha",F.to_date(F.col("fecha")))
        .withColumn("duracion_min",F.col("duracion_min").cast("int"))
        .dropDuplicates(["vuelo_id"])
)

aeronaves_slv=(
    aeronaves_bz
        .select("aeronave_id", "modelo", "fabricante", "anio_fabricacion")
        .withColumn("anio_fabricacion",F.col("anio_fabricacion").cast("int"))
        .dropDuplicates(["aeronave_id"])
)

aeropuertos_slv=(
    aeropuertos_bz
        .select("aeropuerto_id", "nombre", "ciudad", "pais", "lat", "lon")
        .withColumn("lat",F.col("lat").cast("double"))
        .withColumn("lon",F.col("lon").cast("double"))
        .dropDuplicates(["aeropuerto_id"])
)

mantenimientos_slv=(
    mantenimientos_bz
        .select("mantenimiento_id", "aeronave_id", "fecha", "tipo", "costo_usd", "duracion_hr")
        .withColumn("fecha",F.to_date(F.col("fecha")))
        .withColumn("costo_usd",F.col("costo_usd").cast("double"))
        .withColumn("duracion_hr",F.col("duracion_hr").cast("double"))
        .dropDuplicates(["mantenimiento_id"])
)


In [0]:
# Guardar tablas silver
vuelos_slv.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.vuelos") 

aeronaves_slv.write.format("delta") \
.mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.aeronaves")

aeropuertos_slv.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.aeropuertos")

mantenimientos_slv.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", True) \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.mantenimientos")

In [0]:
#Validación de Calidad
print("Nulos vuelo_id:",vuelos_slv.filter(F.col("vuelo_id").isNull()).count())
print("Nulos fecha:",vuelos_slv.filter(F.col("fecha").isNull()).count())
print("Nulos origen_id:",vuelos_slv.filter(F.col("origen_id").isNull()).count())
print("Nulos destino_id:",vuelos_slv.filter(F.col("destino_id").isNull()).count())
print("Nulos aeronave_id:",vuelos_slv.filter(F.col("aeronave_id").isNull()).count())
print("Nulos estado:",vuelos_slv.filter(F.col("estado").isNull()).count())
print("Nulos duracion_min:",vuelos_slv.filter(F.col("duracion_min").isNull()).count())

print("Vuelos duplicados:",vuelos_slv.groupBy("vuelo_id").count().filter(F.col("count") > 1).count())

print("Nulos modelo:",aeronaves_slv.filter(F.col("modelo").isNull()).count())
print("Nulos fabricante:",aeronaves_slv.filter(F.col("fabricante").isNull()).count())
print("Nulos anio_fabricacion:",aeronaves_slv.filter(F.col("anio_fabricacion").isNull()).count())

print("aeronaves duplicadas:",aeronaves_slv.groupBy("aeronave_id").count().filter(F.col("aeronave_id")>1).count())

print("Nulos aeropuerto_id:",aeropuertos_slv.filter(F.col("aeropuerto_id").isNull()).count())
print("Nulos nombre:",aeropuertos_slv.filter(F.col("nombre").isNull()).count())
print("Nulos ciudad:",aeropuertos_slv.filter(F.col("ciudad").isNull()).count())
print("Nulos pais:",aeropuertos_slv.filter(F.col("pais").isNull()).count())
print("Nulos lat:",aeropuertos_slv.filter(F.col("lat").isNull()).count())
print("Nulos lon:",aeropuertos_slv.filter(F.col("lon").isNull()).count())

print("aeropuertos duplicados:",aeropuertos_slv.groupBy("aeropuerto_id").count().filter(F.col("aeropuerto_id")>1).count())

print("Nulos mantenimiento_id:",mantenimientos_slv.filter(F.col("mantenimiento_id").isNull()).count())
print("Nulos aeronave_id:",mantenimientos_slv.filter(F.col("aeronave_id").isNull()).count())
print("Nulos fecha:",mantenimientos_slv.filter(F.col("fecha").isNull()).count())
print("Nulos costo_usd:",mantenimientos_slv.filter(F.col("costo_usd").isNull()).count())
print("Nulos duracion_hr:",mantenimientos_slv.filter(F.col("duracion_hr").isNull()).count())

print("Mantenimientos duplicados:",mantenimientos_slv.groupBy("mantenimiento_id").count().filter(F.col("mantenimiento_id")>1).count())

Nulos vuelo_id: 0
Nulos fecha: 0
Nulos origen_id: 0
Nulos destino_id: 0
Nulos aeronave_id: 0
Nulos estado: 0
Nulos duracion_min: 0
Vuelos duplicados: 0
Nulos modelo: 0
Nulos fabricante: 0
Nulos anio_fabricacion: 0


---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-7382276868459181>, line 16
     13 print("Nulos fabricante:",aeronaves_slv.filter(F.col("fabricante").isNull()).count())
     14 print("Nulos anio_fabricacion:",aeronaves_slv.filter(F.col("anio_fabricacion").isNull()).count())
---> 16 print("aeronaves duplicadas:",aeronaves_slv.groupBy("aeronave_id").count().filter(F.col("aeronave_id")>1).count())
     18 print("Nulos aeropuerto_id:",aeropuertos_slv.filter(F.col("aeropuerto_id").isNull()).count())
     19 print("Nulos nombre:",aeropuertos_slv.filter(F.col("nombre").isNull()).count())

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    31

In [0]:
spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.vuelos").display()
spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.aeronaves").display()
spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.aeropuertos").display()
spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.mantenimientos").display()


vuelo_id,fecha,origen_id,destino_id,aeronave_id,estado,duracion_min
V0017,2024-05-07,CDG,CDG,A004,cancelado,571
V0027,2024-05-28,JFK,LHR,A010,a_tiempo,216
V0048,2024-05-06,LHR,JFK,A007,retrasado,525
V0066,2024-05-03,LHR,CDG,A001,a_tiempo,401
V0072,2024-05-22,MAD,CDG,A009,retrasado,695
V0097,2024-05-26,CDG,BCN,A001,cancelado,315
V0008,2024-05-12,JFK,LHR,A005,a_tiempo,530
V0084,2024-05-12,LHR,JFK,A006,cancelado,381
V0086,2024-05-11,LHR,CDG,A009,a_tiempo,256
V0087,2024-05-14,LHR,MAD,A010,cancelado,368


#### 5. Capa Gold - KPI de puntualidad de vuelos
Se genera una tabla analítica con métricas de puntualidad por modelo, fabricante, país de origen y periodo.

In [0]:
#Leer Silver para puntualidad
df_vuelos = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.vuelos")
df_aeropuertos = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.aeropuertos")
df_aeronaves = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.aeronaves")

In [0]:
#Preparar aeropuertos de origen
df_aeropuertos_origen = df_aeropuertos.select(
    F.col("aeropuerto_id").alias("origen_id"),
    F.col("nombre").alias("origen_nombre"),
    F.col("ciudad").alias("origen_ciudad"),
    F.col("pais").alias("origen_pais")
)

In [0]:
#Validación primeras 5 filas
df_aeropuertos_origen.limit(5).display()

origen_id,origen_nombre,origen_ciudad,origen_pais
BCN,Josep Tarradellas Barcelona-El Prat,Barcelona,España
JFK,John F. Kennedy International,Nueva York,EEUU
CDG,Charles de Gaulle,París,Francia
LHR,Heathrow,Londres,Reino Unido
MAD,Adolfo Suárez Madrid-Barajas,Madrid,España


In [0]:
#Enriquecer vuelos
df_vuelos_enriquecido = (
    df_vuelos
        .join(df_aeropuertos_origen, on="origen_id", how="left")
        .join(df_aeronaves, on="aeronave_id", how="left")
        .withColumn("anio", F.year(F.col("fecha")))
        .withColumn("mes", F.month(F.col("fecha")))
)
#Validación primeras 5 filas
df_vuelos_enriquecido.limit(5).display()

aeronave_id,origen_id,vuelo_id,fecha,destino_id,estado,duracion_min,origen_nombre,origen_ciudad,origen_pais,modelo,fabricante,anio_fabricacion,anio,mes
A004,CDG,V0017,2024-05-07,CDG,cancelado,571,Charles de Gaulle,París,Francia,B737,Boeing,2013,2024,5
A010,JFK,V0027,2024-05-28,LHR,a_tiempo,216,John F. Kennedy International,Nueva York,EEUU,B737,Boeing,2012,2024,5
A007,LHR,V0048,2024-05-06,JFK,retrasado,525,Heathrow,Londres,Reino Unido,A350,Airbus,2015,2024,5
A001,LHR,V0066,2024-05-03,CDG,a_tiempo,401,Heathrow,Londres,Reino Unido,B777,Boeing,2013,2024,5
A009,MAD,V0072,2024-05-22,CDG,retrasado,695,Adolfo Suárez Madrid-Barajas,Madrid,España,A320,Airbus,2009,2024,5


In [0]:
#Calcular el KPI  de puntualidad
df_kpi_puntualidad = (
    df_vuelos_enriquecido
        .groupBy("modelo","fabricante","origen_pais","mes","anio")
        .agg(
            F.count("vuelo_id").alias("total_vuelos"),
            F.count(F.when(F.col("estado") == "a_tiempo", True)).alias("vuelos_a_tiempo"),
            F.count(F.when(F.col("estado") == "retrasado", True)).alias("vuelos_retrasados"),
            F.count(F.when(F.col("estado") == "cancelado", True)).alias("vuelos_cancelados"),
            F.round(F.avg("duracion_min"),2).alias("duracion_promedio_min")
        )
        .withColumn("puntualidad_pct", F.round((F.col("vuelos_a_tiempo") / F.col("total_vuelos")*100), 2))
        .withColumn("retraso_pct", F.round((F.col("vuelos_retrasados") / F.col("total_vuelos")*100), 2))
        .withColumn("cancelacion_pct", F.round((F.col("vuelos_cancelados") / F.col("total_vuelos")*100), 2))
        .withColumn("duracion_promedio_hr", F.round(F.col("duracion_promedio_min") / 60, 2))
)
display(df_kpi_puntualidad)

modelo,fabricante,origen_pais,mes,anio,total_vuelos,vuelos_a_tiempo,vuelos_retrasados,vuelos_cancelados,duracion_promedio_min,puntualidad_pct,retraso_pct,cancelacion_pct,duracion_promedio_hr
A350,Airbus,Francia,5,2024,6,2,2,2,490.67,33.33,33.33,33.33,8.18
A320,Airbus,Reino Unido,5,2024,4,3,0,1,338.0,75.0,0.0,25.0,5.63
B777,Boeing,Reino Unido,5,2024,4,4,0,0,266.5,100.0,0.0,0.0,4.44
A350,Airbus,España,5,2024,12,3,6,3,350.5,25.0,50.0,25.0,5.84
E190,Embraer,Francia,5,2024,2,0,1,1,497.0,0.0,50.0,50.0,8.28
E190,Embraer,Reino Unido,5,2024,1,0,1,0,600.0,0.0,100.0,0.0,10.0
B777,Boeing,Francia,5,2024,7,3,1,3,395.43,42.86,14.29,42.86,6.59
B737,Boeing,Francia,5,2024,6,2,0,4,293.33,33.33,0.0,66.67,4.89
A350,Airbus,Reino Unido,5,2024,4,1,2,1,353.5,25.0,50.0,25.0,5.89
B737,Boeing,Reino Unido,5,2024,5,1,3,1,299.2,20.0,60.0,20.0,4.99


In [0]:
#Guardar Gold Puntualidad
df_kpi_puntualidad.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.kpi_puntualidad_vuelos")

In [0]:
spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.kpi_puntualidad_vuelos").limit(5).display()

modelo,fabricante,origen_pais,mes,anio,total_vuelos,vuelos_a_tiempo,vuelos_retrasados,vuelos_cancelados,duracion_promedio_min,puntualidad_pct,retraso_pct,cancelacion_pct,duracion_promedio_hr
A350,Airbus,Francia,5,2024,6,2,2,2,490.67,33.33,33.33,33.33,8.18
A320,Airbus,Reino Unido,5,2024,4,3,0,1,338.0,75.0,0.0,25.0,5.63
B777,Boeing,Reino Unido,5,2024,4,4,0,0,266.5,100.0,0.0,0.0,4.44
A350,Airbus,España,5,2024,12,3,6,3,350.5,25.0,50.0,25.0,5.84
E190,Embraer,Francia,5,2024,2,0,1,1,497.0,0.0,50.0,50.0,8.28


#### 6. Capa Gold - KPI de costos de mantenimiento

Se genera una tabla analítica con métricas de mantenimiento por aeronave, modelo, fabricante, tipo y periodo.

In [0]:
df_mantenimientos = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.mantenimientos")

In [0]:
#Enriquecer mantenimientos

df_mantenimientos_enriquecido = (
    df_mantenimientos
        .join(df_aeronaves, on="aeronave_id", how="left")
        .withColumn("anio", F.year(F.col("fecha")))
        .withColumn("mes", F.month(F.col("fecha")))
)
#Validación primeras 5 filas
df_mantenimientos_enriquecido.limit(5).display()


aeronave_id,mantenimiento_id,fecha,tipo,costo_usd,duracion_hr,modelo,fabricante,anio_fabricacion,anio,mes
A005,M0007,2024-05-29,correctivo,37821.4,4.3,B777,Boeing,2007,2024,5
A006,M0063,2024-05-15,inspección,5922.58,15.3,A350,Airbus,2012,2024,5
A005,M0079,2024-05-12,correctivo,21684.53,6.9,B777,Boeing,2007,2024,5
A003,M0084,2024-05-23,inspección,25534.6,21.5,E190,Embraer,2020,2024,5
A008,M0092,2024-05-28,correctivo,10407.79,17.4,A350,Airbus,2022,2024,5


In [0]:
#Calcular el KPI  de mantenimiento
df_kpi_mantenimientos = (
    df_mantenimientos_enriquecido
        .groupBy("aeronave_id","modelo","fabricante","tipo","mes","anio")
        .agg(
            F.count("mantenimiento_id").alias("total_mantenimientos"),
            F.round(F.avg("costo_usd"),2).alias("costo_promedio_usd"),
            F.round(F.sum("costo_usd"),2).alias("costo_total_usd"),
            F.round(F.avg("duracion_hr"),2).alias("duracion_promedio_hr"),
            F.round(F.sum("duracion_hr"),2).alias("duracion_total_hr")
        )
        .withColumn("costo_promedio_hr", F.round(F.col("costo_promedio_usd") / F.col("duracion_promedio_hr"), 2))
)
#Validación
display(df_kpi_mantenimientos)

aeronave_id,modelo,fabricante,tipo,mes,anio,total_mantenimientos,costo_promedio_usd,costo_total_usd,duracion_promedio_hr,duracion_total_hr,costo_promedio_hr
A010,B737,Boeing,correctivo,5,2024,3,36064.58,108193.73,9.5,28.5,3796.27
A009,A320,Airbus,preventivo,5,2024,4,15293.45,61173.78,15.33,61.3,997.62
A007,A350,Airbus,preventivo,5,2024,2,14911.75,29823.5,6.15,12.3,2424.67
A002,B737,Boeing,inspección,5,2024,1,44255.92,44255.92,13.8,13.8,3206.95
A001,B777,Boeing,correctivo,5,2024,5,28187.36,140936.81,17.74,88.7,1588.92
A007,A350,Airbus,inspección,5,2024,6,40718.73,244312.38,12.02,72.1,3387.58
A009,A320,Airbus,inspección,5,2024,3,13276.73,39830.19,13.2,39.6,1005.81
A010,B737,Boeing,inspección,5,2024,2,37805.7,75611.39,3.1,6.2,12195.39
A004,B737,Boeing,inspección,5,2024,1,20530.13,20530.13,5.8,5.8,3539.68
A006,A350,Airbus,correctivo,5,2024,2,37393.67,74787.34,14.05,28.1,2661.47


In [0]:
#Guardamos Gold - KPI Mantenimientos
df_kpi_mantenimientos.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.kpi_costo_mantenimiento")
spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.kpi_costo_mantenimiento").limit(5).display()

aeronave_id,modelo,fabricante,tipo,mes,anio,total_mantenimientos,costo_promedio_usd,costo_total_usd,duracion_promedio_hr,duracion_total_hr,costo_promedio_hr
A010,B737,Boeing,correctivo,5,2024,3,36064.58,108193.73,9.5,28.5,3796.27
A009,A320,Airbus,preventivo,5,2024,4,15293.45,61173.78,15.33,61.3,997.62
A007,A350,Airbus,preventivo,5,2024,2,14911.75,29823.5,6.15,12.3,2424.67
A002,B737,Boeing,inspección,5,2024,1,44255.92,44255.92,13.8,13.8,3206.95
A001,B777,Boeing,correctivo,5,2024,5,28187.36,140936.81,17.74,88.7,1588.92
